In [ ]:
!pip install -q timm onnx onnxruntime kaggle --upgrade
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_f2115a9a98fe4a706124b0774716f9f2'

!kaggle competitions download -c aptos2019-blindness-detection -p /content/data
!unzip -q /content/data/aptos2019-blindness-detection.zip -d /content/data/aptos

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 33.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
Mounted at /content/drive
100% 9.51G/9.51G [01:23<00:00, 122MB/s]



In [5]:
import os
os.makedirs('/content/data', exist_ok=True)
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_f2115a9a98fe4a706124b0774716f9f2'
!kaggle competitions download -c aptos2019-blindness-detection -p /content/data
!unzip -q /content/data/aptos2019-blindness-detection.zip -d /content/data/aptos

100% 9.51G/9.51G [01:16<00:00, 134MB/s]



In [ ]:
!kaggle competitions download -c aptos2019-blindness-detection -p /content/data
!unzip -q /content/data/aptos2019-blindness-detection.zip -d /content/data/aptos

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
unzip:  cannot find or open /content/data/aptos2019-blindness-detection.zip, /content/data/aptos2019-blindness-detection.zip.zip or /content/data/aptos2019-blindness-detection.zip.ZIP.


In [6]:
# Cell 2: dataset + dataloader
import pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

labels_df = pd.read_csv('/content/data/aptos/train.csv')

class APTOSDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df, self.img_dir, self.transform = df, img_dir, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.img_dir}/{row['id_code']}.png").convert('RGB')
        return self.transform(img), int(row['diagnosis'])

train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_df = labels_df.sample(frac=0.85, random_state=42)
val_df = labels_df.drop(train_df.index)

train_ds = APTOSDataset(train_df, '/content/data/aptos/train_images', train_tf)
val_ds = APTOSDataset(val_df, '/content/data/aptos/train_images', train_tf)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=32, num_workers=2)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

Train: 3113, Val: 549


In [7]:
# Cell 3: model + training loop with per-epoch checkpointing
import timm, torch.nn as nn

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(15):
    model.train()
    for imgs, labels in train_dl:
        imgs, labels = imgs.cuda(), labels.cuda()
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.cuda(), labels.cuda()
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct/total
    print(f"Epoch {epoch}: val_acc={acc:.4f}")

    torch.save({'epoch': epoch, 'model': model.state_dict(), 'acc': acc},
               f'{CKPT_DIR}/epoch_{epoch}.pt')

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 0: val_acc=0.8051
Epoch 1: val_acc=0.7869
Epoch 2: val_acc=0.8251
Epoch 3: val_acc=0.8233
Epoch 4: val_acc=0.8215


KeyboardInterrupt: 

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
print(os.path.exists('/content/drive/MyDrive'))
print(os.listdir('/content/drive/MyDrive'))

Mounted at /content/drive
True
['Getting started.pdf', 'Classroom', 'CLASS 7 HY SYLLABUS-converted (1)-1.pdf', 'IMG_20210209_151701361.jpg', 'AAAAAAAAAAA.xlsx', 'PPT ON SPREADSHEETS (2).ppt', 'PPT ON SPREADSHEETS (1).ppt', 'PPT ON SPREADSHEETS.ppt', 'PPT ON SPREADSHEETS111.ppt', 'XCELSHEET.xlsx', 'a11.ppt', 'Screenshot_20250911-104148_1.jpg', 'IMG-20250910-WA0002_1.jpg', 'IMG-20250901-WA0050.jpg', 'Students Complete Data 1st year 2025 admission.xlsx', 'Screenshot_20251003-200631.jpg', 'Colab Notebooks', 'PostureAI.gdoc', 'Document from ~dron', '25294917039_dron.pdf', 'goibibio_jalaastra.jpeg', 'sih file']


In [ ]:
!ls -la /content/data/

total 9975888
drwxr-xr-x 3 root root        4096 Sep 20 06:08 .
drwxr-xr-x 1 root root        4096 Sep 20 06:07 ..
drwxr-xr-x 4 root root        4096 Sep 20 06:08 aptos
-rw-r--r-- 1 root root 10215289875 Dec 18  2019 aptos2019-blindness-detection.zip


In [ ]:
!ls /content/data/aptos
!head -5 /content/data/aptos/train.csv
!ls /content/data/aptos/train_images | wc -l

sample_submission.csv  test.csv  test_images  train.csv  train_images
id_code,diagnosis
000c1434d8d7,2
001639a390f0,4
0024cdab0c1e,1
002c21358ce6,0
3662


In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

True Tesla T4


In [3]:
import os
CKPT_DIR = '/content/drive/MyDrive/sih_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoint dir exists:", os.path.exists(CKPT_DIR))

Checkpoint dir exists: True


In [8]:
from google.colab import files
files.download('/content/data/aptos/train_images/000c1434d8d7.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import os
CKPT_DIR = '/content/drive/MyDrive/sih_checkpoints'
print(sorted(os.listdir(CKPT_DIR)))

['epoch_0.pt', 'epoch_1.pt', 'epoch_2.pt', 'epoch_3.pt', 'epoch_4.pt']


In [2]:
import os
CKPT_DIR = '/content/drive/MyDrive/sih_checkpoints'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import torch, timm
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=5)
ckpt = torch.load(f'{CKPT_DIR}/epoch_1.pt', map_location='cpu')
model.load_state_dict(ckpt['model'])
model.eval()

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2

In [16]:
dummy_input = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, dummy_input,
    f'{CKPT_DIR}/dr_grade.onnx',
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    dynamo=False
)
print("Exported to", f'{CKPT_DIR}/dr_grade.onnx')

/tmp/ipykernel_1110/2720900004.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


VersionError: Detected incompatible Protobuf Gencode/Runtime versions when loading onnx/onnx-ml.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.

In [5]:
dummy_input = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, dummy_input,
    f'{CKPT_DIR}/dr_grade.onnx',
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    dynamo=False
)
print("Exported to", f'{CKPT_DIR}/dr_grade.onnx')

/tmp/ipykernel_19987/2720900004.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported to /content/drive/MyDrive/sih_checkpoints/dr_grade.onnx


In [6]:
from google.colab import files
files.download(f'{CKPT_DIR}/dr_grade.onnx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import os
os.makedirs('/content/data', exist_ok=True)
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_f2115a9a98fe4a706124b0774716f9f2'
!kaggle competitions download -c aptos2019-blindness-detection -p /content/data
!unzip -q /content/data/aptos2019-blindness-detection.zip -d /content/data/aptos

100% 9.51G/9.51G [01:20<00:00, 128MB/s]



In [3]:
import shutil, os
os.makedirs('/content/drive/MyDrive/sih_checkpoints', exist_ok=True)
shutil.copy('/content/data/aptos2019-blindness-detection.zip', '/content/drive/MyDrive/sih_checkpoints/aptos2019-blindness-detection.zip')
print("Backed up to Drive")

Backed up to Drive


In [5]:
import random, shutil, os
from google.colab import files

random.seed(42)
img_dir = '/content/data/aptos/train_images'
sample_files = random.sample(os.listdir(img_dir), 8)

os.makedirs('/content/quality_samples', exist_ok=True)
for f in sample_files:
    shutil.copy(f'{img_dir}/{f}', f'/content/quality_samples/{f}')

shutil.make_archive('/content/quality_samples', 'zip', '/content/quality_samples')
print("Sample files:", sample_files)
files.download('/content/quality_samples.zip')

Sample files: ['a5c9a8c726b2.png', 'bf8092e4001d.png', 'ab3c505b624f.png', '8af50c9d0a86.png', 'eda29a9d78f3.png', '357f02a779d7.png', 'f71bea807c96.png', 'be7bc89f5fec.png']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>